# 📑 THỬ NGHIỆM PHÁT HIỆN MÂU THUẪN (PHIÊN BẢN 2)
Tài liệu này tập trung vào việc tinh chỉnh các tham số và thử nghiệm các mô hình NLI nâng cao trên tập dữ liệu dài.


## I. Cấu hình & Import Thư viện


In [ ]:
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from typing import List, Dict, Tuple
import argparse, os, json, pickle
import faiss
import json
import numpy as np
import os
import re
import shutil
import torch
import torch.nn.functional as F


## II. Chuẩn bị môi trường Kaggle


In [12]:

warnings.filterwarnings("ignore")
sys.tracebacklimit = 0
logging.disable(logging.warning)

# -----------------------------
# 1. copy model về thư mục working và sửa config modernbert
# -----------------------------
src_model_path = "/kaggle/input/model5"
working_model_path = "/kaggle/working/model5"
working_model_path1 = "/kaggle/working/model5/transformers/default/1"

shutil.copytree(src_model_path, working_model_path, dirs_exist_ok=true)

config_file = os.path.join(working_model_path1, "config.json")
with open(config_file, "r") as f:
    cfg = json.load(f)
cfg["model_type"] = "modernbert"
with open(config_file, "w") as f:
    json.dump(cfg, f)

# -----------------------------
# 2. load modernbert nli model
# -----------------------------
tokenizer = autotokenizer.from_pretrained(working_model_path1, local_files_only=true, local_files_only=True)
config = autoconfig.from_pretrained(working_model_path1, local_files_only=true, local_files_only=True)
model = automodelforsequenceclassification.from_pretrained(
    working_model_path1, config=config, local_files_only=true
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# -----------------------------
# 3. load falcon-h1-3b-instruct via llama_cpp
# -----------------------------
llm = llama.from_pretrained(
    repo_id="tiiuae/falcon-h1-3b-instruct-gguf",
    filename="falcon-h1-3b-instruct-bf16.gguf",
)

# -----------------------------
# 4. hàm tạo summary từ falcon-h1-3b
# -----------------------------
def generate_hypothesis_summary(text):
    prompt = (
        "summarize all the actions the suspect has performed in this block based on the question-answer conversation. "
        "focus on main actions, ignore emotions and unnecessary words.\n\n"
        f"{text}\n\nsummary of actions:"
    )
    out = llm(prompt, max_tokens=100, stop=["\n"])
    summary = out["choices"][0]["text"].strip()
    return summary

# -----------------------------
# 5. hàm chạy nli cho cặp (premise, hypothesis)
# -----------------------------
def run_nli(premise, hypothesis):
    inputs = tokenizer(
        premise, hypothesis, return_tensors="pt", truncation=true, max_length=512, padding=true
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return {
        "entailment": float(probs[0]),
        "neutral": float(probs[1]),
        "contradiction": float(probs[2]),
    }

# -----------------------------
# 6. chia premise thành spans theo từ
# -----------------------------
def split_into_spans(text, max_len=50):
    words = text.split()
    spans = [" ".join(words[i:i+max_len]) for i in range(0, len(words), max_len)]
    return spans

# -----------------------------
# 7. chia hypothesis theo block interrogator -> suspect
# -----------------------------
def split_qa_blocks(text):
    lines = text.splitlines()
    blocks = []
    current_block = ""
    for line in lines:
        line = line.strip()
        if line.startswith("interrogator:"):
            if current_block:
                blocks.append(current_block.strip())
            current_block = line
        elif line.startswith("suspect:"):
            current_block += " " + line
    if current_block:
        blocks.append(current_block.strip())
    return blocks

# -----------------------------
# 8. multi-hop concat và chain
# -----------------------------
def multi_hop_concat(spans, hypothesis):
    concat_premise = " ".join(spans)
    return run_nli(concat_premise, hypothesis)

def multi_hop_chain(spans, hypothesis):
    results = []
    for i in range(len(spans)):
        for j in range(i+1, len(spans)):
            pair_premise = spans[i] + " " + spans[j]
            results.append(((i, j), run_nli(pair_premise, hypothesis)))
    return results

# -----------------------------
# 9. ví dụ dữ liệu
# -----------------------------
premise = """on 4 january 1989, after losing money in a game of mahjong the night before, 
miyano decided to take his anger out on furuta. he ignited a candle and dripped hot wax on her face, 
placed two shortened candles on her eyelids, and forced her to drink her own urine. 
furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. 
to prevent them from being stained with blood, the group covered their hands in plastic bags 
before beating her with their fists and an iron exercise ball, and dropped the ball on her abdomen several times. 
miyano poured lighter fluid on furuta and set her on fire; she made weak attempts to put herself out, 
but soon stopped moving. the assault lasted for about two hours, after which furuta died at 10 a.m."""

hypothesis = """interrogator: on the night of january 4th, what were you doing after the mahjong game?
suspect: i was angry about losing, but i don’t remember doing anything extreme.
interrogator: several witnesses indicate that you poured hot wax on furuta’s face and placed candles on her eyelids. can you explain why you would do that?
suspect: i… i don’t recall that.
interrogator: after that, it is reported that she was forced to drink her own urine, kicked, and fell onto a stereo unit, leading to convulsions. why would you inflict such acts on her?
suspect: i… i don’t know…
interrogator: there are reports that you and others covered your hands in plastic bags and beat her with fists and an iron exercise ball, then set her on fire using lighter fluid. she died as a result. can you tell me what happened during those two hours?
suspect: i… i can’t remember…
interrogator: do you understand that your actions directly caused her death at 10 a.m.?"""

# -----------------------------
# 10. chia premise & hypothesis thành spans/blocks
# -----------------------------
premise_spans = split_into_spans(premise, max_len=40)
hypothesis_blocks = split_qa_blocks(hypothesis)

# tạo summary cho từng block
hypothesis_summaries = []
for block in hypothesis_blocks:
    try:
        summary = generate_hypothesis_summary(block)
        hypothesis_summaries.append({"original": block, "summary": summary})
    except runtimeerror:
        hypothesis_summaries.append({"original": block, "summary": ""})

# -----------------------------
# 11. tính nli
# -----------------------------
rerank_results = []
for s, hyp in zip(premise_spans, hypothesis_summaries):
    probs = run_nli(s, hyp["summary"])
    rerank_results.append({
        "premise_span": s,
        "hypothesis_summary": hyp["summary"],
        "hypothesis_original": hyp["original"],
        "probs": probs
    })

concat_probs = multi_hop_concat(premise_spans, " ".join([h["summary"] for h in hypothesis_summaries]))
chain_results = multi_hop_chain(premise_spans, " ".join([h["summary"] for h in hypothesis_summaries]))

# -----------------------------
# 12. tổng hợp kết quả
# -----------------------------
entail_key = "entailment"
neutral_key = "neutral"
contrad_key = "contradiction"

best_span = max(rerank_results, key=lambda r: r["probs"][entail_key])
mean_entail = np.mean([r["probs"][entail_key] for r in rerank_results])

best_chain = none
if chain_results:
    best_chain = max(chain_results, key=lambda r: r[1][entail_key])

# -----------------------------
# 13. in kết quả
# -----------------------------
print("=== single spans comparison ===")
for idx, r in enumerate(rerank_results, 1):
    print(f"# {idx}")
    print("premise span:\n", r["premise_span"])
    print("hypothesis original:\n", r["hypothesis_original"])
    print("hypothesis summary:\n", r["hypothesis_summary"])
    print("probs -> entailment: {:.3f} | neutral: {:.3f} | contradiction: {:.3f}".format(
        r["probs"][entail_key], r["probs"][neutral_key], r["probs"][contrad_key]))
    print("-"*60)

print("\n=== multi-hop concat ===")
print("premise (all spans concatenated):\n", " ".join(premise_spans))
print("hypothesis summary (all concatenated):\n", " ".join([h["summary"] for h in hypothesis_summaries]))
print("probs -> entailment: {:.3f} | neutral: {:.3f} | contradiction: {:.3f}".format(
    concat_probs[entail_key], concat_probs[neutral_key], concat_probs[contrad_key]))

if best_chain:
    i, j = best_chain[0]
    probs = best_chain[1]
    print("\n=== best multi-hop chain ===")
    print("spans indices:", i, j)
    print("span texts:\n", premise_spans[i], "\n---\n", premise_spans[j])
    print("probs -> entailment: {:.3f} | neutral: {:.3f} | contradiction: {:.3f}".format(
        probs[entail_key], probs[neutral_key], probs[contrad_key]))

print("\n=== best single span (highest entailment) ===")
print("premise span:\n", best_span["premise_span"])
print("hypothesis original:\n", best_span["hypothesis_original"])
print("hypothesis summary:\n", best_span["hypothesis_summary"])
print("probs -> entailment: {:.3f} | neutral: {:.3f} | contradiction: {:.3f}".format(
    best_span["probs"][entail_key], best_span["probs"][neutral_key], best_span["probs"][contrad_key]))

print("\nmean entail (single spans):", mean_entail)


./Falcon-H1-3B-Instruct-BF16.gguf:   0%|          | 0.00/6.30G [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 41 key-value pairs and 547 tensors from /root/.cache/huggingface/hub/models--tiiuae--Falcon-H1-3B-Instruct-GGUF/snapshots/18cc9812739f6040f795ddf9d92c9da9a8551572/./Falcon-H1-3B-Instruct-BF16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = falcon-h1
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Falcon H1 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Falcon-H1
llama_model_loader: - kv   5:                         general.size_label str              = 3B
llama_model_loader: - kv

=== Single spans comparison ===
# 1
Premise span:
 On 4 January 1989, after losing money in a game of mahjong the night before, Miyano decided to take his anger out on Furuta. He ignited a candle and dripped hot wax on her face, placed two shortened candles on
Hypothesis original:
 Interrogator: On the night of January 4th, what were you doing after the mahjong game? Suspect: I was angry about losing, but I don’t remember doing anything extreme.
Hypothesis summary:
 Played mahjong, angry about losing. I don't remember anything extreme.
Probs -> Entailment: 0.000 | Neutral: 0.005 | Contradiction: 0.995
------------------------------------------------------------
# 2
Premise span:
 her eyelids, and forced her to drink her own urine. Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. To prevent them from being stained with blood, the group covered their hands in
Hypothesis original:
 Interrogator: Several witnesses indicate that you poured hot wax on Fu

In [1]:

# -----------------------------
# 1. copy model về thư mục working và sửa config
# -----------------------------
src_model_path = "/kaggle/input/model5"
working_model_path = "/kaggle/working/model5"
working_model_path1 = "/kaggle/working/model5/transformers/default/1"

shutil.copytree(src_model_path, working_model_path, dirs_exist_ok=true)

# sửa config.json để automodel nhận ra modernbert
config_file = os.path.join(working_model_path1, "config.json")
with open(config_file, "r") as f:
    cfg = json.load(f)
cfg["model_type"] = "modernbert"
with open(config_file, "w") as f:
    json.dump(cfg, f)

# -----------------------------
# 2. load tokenizer và model bằng auto
# -----------------------------
tokenizer = autotokenizer.from_pretrained(working_model_path1, local_files_only=true, local_files_only=True)
config = autoconfig.from_pretrained(working_model_path1, local_files_only=true, local_files_only=True)
model = automodelforsequenceclassification.from_pretrained(working_model_path1, config=config, local_files_only=true, local_files_only=True)
model.eval()

# -----------------------------
# 3. hàm chạy nli cho 1 cặp (premise, hypothesis)
# -----------------------------
def run_nli(premise, hypothesis):
    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=true, max_length=512, padding=true)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return {
        "entailment": float(probs[0]),
        "neutral": float(probs[1]),
        "contradiction": float(probs[2]),
    }

# -----------------------------
# 4. chia văn bản thành spans
# -----------------------------
def split_into_spans(text, max_len=50):
    words = text.split()
    spans = []
    for i in range(0, len(words), max_len):
        spans.append(" ".join(words[i:i+max_len]))
    return spans

# -----------------------------
# 5. multi-hop concat: gộp tất cả spans làm premise
# -----------------------------
def multi_hop_concat(spans, hypothesis):
    concat_premise = " ".join(spans)
    return run_nli(concat_premise, hypothesis)

# -----------------------------
# 6. multi-hop chain: xét từng cặp spans
# -----------------------------
def multi_hop_chain(spans, hypothesis):
    results = []
    for i in range(len(spans)):
        for j in range(i+1, len(spans)):
            pair_premise = spans[i] + " " + spans[j]
            results.append(((i, j), run_nli(pair_premise, hypothesis)))
    return results

# -----------------------------
# 7. ví dụ chạy thử
# -----------------------------
premise = """
on 4 january 1989, after losing money in a game of mahjong the night before, 
miyano decided to take his anger out on furuta. he ignited a candle and dripped hot wax on her face, 
placed two shortened candles on her eyelids, and forced her to drink her own urine. 
furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. 
to prevent them from being stained with blood, the group covered their hands in plastic bags 
before beating her with their fists and an iron exercise ball, and dropped the ball on her abdomen several times. 
miyano poured lighter fluid on furuta and set her on fire; she made weak attempts to put herself out, 
but soon stopped moving. the assault lasted for about two hours, after which furuta died at 10 a.m.
"""

hypothesis = """furuta was tortured until she died."""

# chia premise thành spans
spans = split_into_spans(premise, max_len=40)
print("số spans:", len(spans))

# tính nli cho từng span riêng lẻ
rerank_results = []
for s in spans:
    rerank_results.append({"span": s, "probs": run_nli(s, hypothesis)})

# multi-hop concat
concat_probs = multi_hop_concat(spans, hypothesis)

# multi-hop chain (cặp spans)
chain_results = multi_hop_chain(spans, hypothesis)

# -----------------------------
# 8. tổng hợp kết quả
# -----------------------------
entail_key = "entailment"
best_span = max(rerank_results, key=lambda r: r["probs"][entail_key])
mean_entail = np.mean([r["probs"][entail_key] for r in rerank_results])

best_chain = none
if chain_results:
    best_chain = max(chain_results, key=lambda r: r[1][entail_key])

print("\n=== kết quả ===")
print("best single span entail:", best_span["probs"][entail_key])
if best_chain:
    print("best pair chain entail:", best_chain[1][entail_key], " -- spans:", best_chain[0])
else:
    print("best pair chain entail: n/a (not enough spans)")
print("concat entail:", concat_probs[entail_key])
print("mean entail (single spans):", mean_entail)

print("\nbest single span text:\n", best_span["span"])
if best_chain:
    print("\nbest chain spans text:\n", spans[best_chain[0][0]], "\n---\n", spans[best_chain[0][1]])


2025-09-27 00:21:28.930644: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758932489.154088      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758932489.215576      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Số spans: 4

=== Kết quả ===
Best single span entail: 0.9469957947731018
Best pair chain entail: 0.9999854564666748  -- spans: (1, 3)
Concat entail: 0.9999693632125854
Mean entail (single spans): 0.2527297168271616

Best single span text:
 out, but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.

Best chain spans text:
 her eyelids, and forced her to drink her own urine. Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. To prevent them from being stained with blood, the group covered their hands in 
---
 out, but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.


## III. Thử nghiệm Baseline (DeBERTa/MiniLM)


In [ ]:
#!/usr/bin/env python3
"""
document contradiction detection with deberta-small long nli
compare two long documents and find contradictory statements
"""


class documentcontradictiondetector:
    def __init__(self, model_name=working_model_path1):
        """
        initialize with deberta-small long nli model for document comparison
        """
        print("loading deberta-small long nli model...")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # load deberta-small for nli
        self.tokenizer = autotokenizer.from_pretrained(model_name)
        self.model = automodelforsequenceclassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        
        print(f"model loaded on device: {self.device}")
    
    def split_into_sentences(self, text: str) -> list[str]:
        """simple sentence splitting"""
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 15]
        return sentences
    
    def get_contradiction_score(self, text1: str, text2: str) -> tuple[float, float, float]:
        """
        get nli score between two texts
        returns (contradiction, neutral, entailment) probabilities
        """
        inputs = self.tokenizer(
            text1,
            text2,
            truncation=true,
            padding=true,
            max_length=512,
            return_tensors="pt"
        )
        
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = f.softmax(outputs.logits, dim=-1)
            
            # return (contradiction, neutral, entailment) scores
            return tuple(probs[0].cpu().numpy())
    
    def find_contradictions_semantic_matching(self, doc1: str, doc2: str,
                                            contradiction_threshold: float = 0.5) -> list[dict]:
        """
        smart approach: first find semantically similar sentences, then check for contradictions
        """
        sentences1 = self.split_into_sentences(doc1)
        sentences2 = self.split_into_sentences(doc2)
        
        print(f"smart matching: {len(sentences1)} vs {len(sentences2)} sentences...")
        
        contradictions = []
        
        # for each sentence in doc1, find potentially related sentences in doc2
        for i, sent1 in enumerate(sentences1):
            # look for sentences in doc2 that might be talking about similar topics
            words1 = set(sent1.lower().split())
            
            candidates = []
            for j, sent2 in enumerate(sentences2):
                words2 = set(sent2.lower().split())
                # calculate simple word overlap
                overlap = len(words1.intersection(words2))
                if overlap >= 2:  # at least 2 common words
                    candidates.append((j, sent2))
            
            # check nli for promising candidates
            for j, sent2 in candidates:
                contradiction, neutral, entailment = self.get_contradiction_score(sent1, sent2)
                
                if contradiction > contradiction_threshold:
                    contradictions.append({
                        'doc1_sentence': sent1,
                        'doc2_sentence': sent2,
                        'doc1_index': i,
                        'doc2_index': j,
                        'contradiction_score': float(contradiction),
                        'neutral_score': float(neutral),
                        'entailment_score': float(entailment),
                        'word_overlap': len(words1.intersection(set(sent2.lower().split())))
                    })
        
        # sort by contradiction score
        contradictions.sort(key=lambda x: x['contradiction_score'], reverse=true)
        
        return contradictions
    
  

In [ ]:


class documentcontradictiondetector:
    def __init__(self):
        self.model = sentencetransformer('sentence-transformers/paraphrase-minilm-l6-v2')

    
    
    def split_into_sentences(self, doc: str) -> list[str]:
            return [
                sentence.strip()
                for sentence in doc.strip().split('.')
                if sentence.strip() and not sentence.strip().lower().startswith('interrogator:')
            ]
 

    def find_contradictions_semantic_matching(self, doc1: str, doc2: str) -> list[dict]:
        sentences1 = self.split_into_sentences(doc1)
        sentences2 = self.split_into_sentences(doc2)

        embeddings1 = self.model.encode(sentences1, convert_to_tensor=true)
        embeddings2 = self.model.encode(sentences2, convert_to_tensor=true)

        contradiction_results = []

        for i, emb1 in enumerate(embeddings1):
            cosine_scores = util.pytorch_cos_sim(emb1, embeddings2)[0]
            for j, score in enumerate(cosine_scores):
                similarity = score.item()
                contradiction_score = 1 - similarity  # càng khác thì điểm mâu thuẫn càng cao
                if contradiction_score > 0.4:  # ngưỡng có thể điều chỉnh
                    contradiction_results.append({
                        'doc1_sentence': sentences1[i],
                        'doc2_sentence': sentences2[j],
                        'similarity_score': similarity,
                        'contradiction_score': contradiction_score,
                        'entailment_score': 1 - contradiction_score,  # mô phỏng
                        'neutral_score': 0.5  # dummy, bạn có thể tích hợp mô hình nli thật
                    })

        # sắp xếp theo mức độ mâu thuẫn giảm dần
        contradiction_results.sort(key=lambda x: x['contradiction_score'], reverse=true)
        return contradiction_results

    def extract_contradictory_segments(self, doc1: str, doc2: str, max_contradictions: int = 10) -> dict:
        contradictions = self.find_contradictions_semantic_matching(doc1, doc2)
        contradictions = contradictions[:max_contradictions]

        if contradictions:
            avg_contradiction_score = np.mean([c['contradiction_score'] for c in contradictions])
            max_contradiction_score = max([c['contradiction_score'] for c in contradictions])
        else:
            avg_contradiction_score = 0
            max_contradiction_score = 0

        return {
            'contradictions': contradictions,
            'total_found': len(contradictions),
            'avg_contradiction_score': float(avg_contradiction_score),
            'max_contradiction_score': float(max_contradiction_score),
            'doc1_sentences': len(self.split_into_sentences(doc1)),
            'doc2_sentences': len(self.split_into_sentences(doc2))
        }

    def generate_contradiction_report(self, doc1: str, doc2: str, max_contradictions: int = 5) -> str:
        result = self.extract_contradictory_segments(doc1, doc2, max_contradictions)

        report = f"""=== contradiction analysis report ===

documents analyzed:
document 1: {result['doc1_sentences']} sentences
document 2: {result['doc2_sentences']} sentences

contradictions found: {result['total_found']}
average contradiction score: {result['avg_contradiction_score']:.3f}
highest contradiction score: {result['max_contradiction_score']:.3f}
"""

        if result['contradictions']:
            report += "=== top contradictions ===\n\n"
            for i, contradiction in enumerate(result['contradictions'], 1):
                report += f"contradiction #{i} (score: {contradiction['contradiction_score']:.3f})\n"
                report += f"doc1: {contradiction['doc1_sentence']}\n"
                report += f"doc2: {contradiction['doc2_sentence']}\n"
                report += f"entailment: {contradiction['entailment_score']:.3f} | neutral: {contradiction['neutral_score']:.3f}\n"
                report += "-" * 80 + "\n\n"
        else:
            report += "no significant contradictions found.\n"

        return report

## IV. Xây dựng Hệ thống Retrieval (FAISS Indexing)


In [ ]:

def split_spans(text, max_sent=6):
    sents = sent_tokenize(text)
    spans = [" ".join(sents[i:i+max_sent]) for i in range(0, len(sents), max_sent)]
    return spans

def main():
    parser = argparse.argumentparser()
    parser.add_argument("--input", required=true, help="input corpus jsonl, each line: {id, text}")
    parser.add_argument("--outdir", default="./index_modernbert", help="output dir")
    parser.add_argument("--embed_model", default="nomic-ai/modernbert-embed-base", help="modernbert embed model")
    args = parser.parse_args()

    os.makedirs(args.outdir, exist_ok=true)

    model = sentencetransformer(args.embed_model)
    spans, id_map = [], []
    with open(args.input, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            doc_id, text = obj["id"], obj["text"]
            for span in split_spans(text):
                spans.append(span)
                id_map.append(doc_id)

    print(f"encoding {len(spans)} spans ...")
    emb = model.encode(spans, batch_size=32, show_progress_bar=true, convert_to_numpy=true, normalize_embeddings=true)

    dim = emb.shape[1]
    index = faiss.indexflatip(dim)  # cosine similarity since normalized
    index.add(emb)

    faiss.write_index(index, os.path.join(args.outdir, "index.faiss"))
    with open(os.path.join(args.outdir, "spans.pkl"), "wb") as f:
        pickle.dump(spans, f)
    with open(os.path.join(args.outdir, "id_map.pkl"), "wb") as f:
        pickle.dump(id_map, f)

    print("index built and saved to", args.outdir)

if __name__ == "__main__":
    main()


## V. Suy luận Nâng cao & Multi-hop Reasoning


In [3]:


# model cross-encoder modernbert
model_name = "tasksource/modernbert-large-nli"
tokenizer = autotokenizer.from_pretrained(model_name)
model = automodelforsequenceclassification.from_pretrained(model_name)
model.eval()

# hàm chạy nli 1 cặp (premise, hypothesis)
def run_nli(premise, hypothesis):
    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=true, max_length=512)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    return {
        "entailment": float(probs[0]),
        "neutral": float(probs[1]),
        "contradiction": float(probs[2]),
    }

# hàm split văn bản thành spans ngắn
def split_into_spans(text, max_len=50):
    words = text.split()
    spans = []
    for i in range(0, len(words), max_len):
        spans.append(" ".join(words[i:i+max_len]))
    return spans

# multi-hop concat: nối nhiều spans lại
def multi_hop_concat(spans, hypothesis):
    concat_premise = " ".join(spans)
    return run_nli(concat_premise, hypothesis)

# multi-hop chain: xét từng cặp spans
def multi_hop_chain(spans, hypothesis):
    results = []
    for i in range(len(spans)):
        for j in range(i+1, len(spans)):
            pair_premise = spans[i] + " " + spans[j]
            results.append(((i, j), run_nli(pair_premise, hypothesis)))
    return results

# =============================
# ví dụ chạy thử
# =============================
premise = """
on 4 january 1989, after losing money in a game of mahjong the night before, 
miyano decided to take his anger out on furuta. he ignited a candle and dripped hot wax on her face, 
placed two shortened candles on her eyelids, and forced her to drink her own urine. 
furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. 
to prevent them from being stained with blood, the group covered their hands in plastic bags 
before beating her with their fists and an iron exercise ball, and dropped the ball on her abdomen several times. 
miyano poured lighter fluid on furuta and set her on fire; she made weak attempts to put herself out, 
but soon stopped moving. the assault lasted for about two hours, after which furuta died at 10 a.m.
"""

hypothesis = "furuta was tortured until she died."

# b1: chia premise thành spans
spans = split_into_spans(premise, max_len=40)
print("số spans:", len(spans))

# b2: tính nli cho từng span riêng lẻ
rerank_results = []
for s in spans:
    rerank_results.append({"span": s, "probs": run_nli(s, hypothesis)})

# b3: multi-hop concat
concat_probs = multi_hop_concat(spans, hypothesis)

# b4: multi-hop chain (cặp spans)
chain_results = multi_hop_chain(spans, hypothesis)

# =============================
# tổng hợp kết quả
# =============================
entail_key = "entailment"
best_span = max(rerank_results, key=lambda r: r["probs"][entail_key])
mean_entail = np.mean([r["probs"][entail_key] for r in rerank_results])

best_chain = none
if chain_results:  # có ít nhất 1 cặp
    best_chain = max(chain_results, key=lambda r: r[1][entail_key])

print("\n=== kết quả ===")
print("best single span entail:", best_span["probs"][entail_key])
if best_chain:
    print("best pair chain entail:", best_chain[1][entail_key], " -- spans:", best_chain[0])
else:
    print("best pair chain entail: n/a (not enough spans)")
print("concat entail:", concat_probs[entail_key])
print("mean entail (single spans):", mean_entail)

print("\nbest single span text:\n", best_span["span"])
if best_chain:
    print("\nbest chain spans text:\n", spans[best_chain[0][0]], "\n---\n", spans[best_chain[0][1]])


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Số spans: 4

=== Kết quả ===
Best single span entail: 0.8224649429321289
Best pair chain entail: 0.9607888460159302  -- spans: (1, 3)
Concat entail: 0.9645508527755737
Mean entail (single spans): 0.25921718147583306

Best single span text:
 out, but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.

Best chain spans text:
 her eyelids, and forced her to drink her own urine. Furuta was lifted and kicked, fell onto a stereo unit, and began a fit of convulsions. To prevent them from being stained with blood, the group covered their hands in 
---
 out, but soon stopped moving. The assault lasted for about two hours, after which Furuta died at 10 a.m.


## VI. Các thử nghiệm khác


In [ ]:
def extract_contradictory_segments(self, doc1: str, doc2: str, 
                                   max_contradictions: int = 10) -> dict:
    """
    main function to extract contradictory segments from two documents
    """
    contradictions = self.find_contradictions_semantic_matching(doc1, doc2)

    # limit results
    contradictions = contradictions[:max_contradictions]

    # create summary
    if contradictions:
        avg_contradiction_score = np.mean([c['contradiction_score'] for c in contradictions])
        max_contradiction_score = max([c['contradiction_score'] for c in contradictions])
    else:
        avg_contradiction_score = 0
        max_contradiction_score = 0

    return {
        'contradictions': contradictions,
        'total_found': len(contradictions),
        'avg_contradiction_score': float(avg_contradiction_score),
        'max_contradiction_score': float(max_contradiction_score),
        'doc1_sentences': len(self.split_into_sentences(doc1)),
        'doc2_sentences': len(self.split_into_sentences(doc2))
    }

def generate_contradiction_report(self, doc1: str, doc2: str, 
                                   max_contradictions: int = 5) -> str:
    """
    generate a readable report of contradictions found
    """
    result = self.extract_contradictory_segments(doc1, doc2, max_contradictions)

    report = f"""=== contradiction analysis report ===

documents analyzed:
document 1: {result['doc1_sentences']} sentences
document 2: {result['doc2_sentences']} sentences

contradictions found: {result['total_found']}
average contradiction score: {result['avg_contradiction_score']:.3f}
highest contradiction score: {result['max_contradiction_score']:.3f}
"""
    if result['contradictions']:
        report += "=== top contradictions ===\n\n"
        
        for i, contradiction in enumerate(result['contradictions'][:max_contradictions], 1):
            report += f"contradiction #{i} (score: {contradiction['contradiction_score']:.3f})\n"
            report += f"doc1: {contradiction['doc1_sentence']}\n"
            report += f"doc2: {contradiction['doc2_sentence']}\n"
            report += f"entailment: {contradiction['entailment_score']:.3f} | neutral: {contradiction['neutral_score']:.3f}\n"
            report += "-" * 80 + "\n\n"
    else:
        report += "no significant contradictions found.\n"

    return report
